# TAHAP 3b - LAPIS 2: Training Model Supervised (Random Forest & XGBoost)

Pada tahap ini kita akan melatih **dua model supervised** untuk perbandingan:
- **Random Forest**: model baseline yang stabil
- **XGBoost**: model advanced dengan performa tinggi

Kedua model akan belajar dari:
- Fitur dasar perilaku kasir
- Skor anomali Isolation Forest (hasil dari Lapis 1)

Penting: `is_fraud` HANYA menjadi target (`y`), TIDAK PERNAH masuk sebagai fitur (`X`).

## Import & Konfigurasi

In [3]:
import os
import sys
import joblib
import warnings

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Coba import XGBoost; jika gagal, berikan peringatan
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("[!] XGBoost belum terpasang. Jalankan: pip install xgboost")

warnings.filterwarnings('ignore')

# Tambahkan path agar bisa import modul dari folder ml_pipeline
current_dir = os.path.abspath('')
if os.path.basename(current_dir) == 'ml_pipeline':
    ml_path = current_dir
else:
    ml_path = os.path.join(current_dir, 'ml_pipeline')
if ml_path not in sys.path:
    sys.path.insert(0, ml_path)

from feature_engineering import load_data
from pola_b_features import build_pola_b_matrix

#  Konfigurasi path model 
RF_MODEL_PATH = os.path.join(ml_path, "models", "supervised_rf.pkl")
XGB_MODEL_PATH = os.path.join(ml_path, "models", "supervised_xgb.pkl")
SPLIT_PATH = os.path.join(ml_path, "models", "test_split.pkl")
RANDOM_STATE = 42

#  Parameter Random Forest 
RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "class_weight": "balanced",   # menangani fraud yang minoritas
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

#   Parameter XGBoost  
XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "eval_metric": "logloss",
}

print("[OK] Import & konfigurasi selesai")
print(f"     XGBoost tersedia: {XGBOOST_AVAILABLE}")

[OK] Import & konfigurasi selesai
     XGBoost tersedia: True


## Load Data & Build Fitur Pola B

In [4]:
print("[1/6] Memuat data + membangun fitur Pola B (fitur dasar + skor IF)...")

# Load data mentah dari database
raw = load_data()

# Bangun matriks fitur Pola B (termasuk if_anomaly_score)
X = build_pola_b_matrix(raw)

# Target = label kunci jawaban (is_fraud)
y = raw["is_fraud"].astype(int)

print(f"      {X.shape[0]:,} baris x {X.shape[1]} fitur | fraud: {int(y.sum())}")
print(f"\n  Kolom fitur yang dipakai:")
for i, col in enumerate(X.columns, 1):
    print(f"    {i:2d}. {col}")

[1/6] Memuat data + membangun fitur Pola B (fitur dasar + skor IF)...
      2,116 baris x 10 fitur | fraud: 116

  Kolom fitur yang dipakai:
     1. hour_of_day
     2. is_refund
     3. time_gap_seconds
     4. txn_freq_daily
     5. refund_count_daily
     6. refund_ratio_daily
     7. amount_zscore_cashier
     8. amount_rolling_mean_5
     9. amount_deviation_from_mean
    10. if_anomaly_score


## Train/Test Split (Stratified 70/30)

In [5]:
print("[2/6] Membagi train/test (stratified 70/30)...")

# Split dengan stratify agar proporsi fraud terjaga di train dan test
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, 
    test_size=0.30, 
    stratify=y, 
    random_state=RANDOM_STATE
)

print(f"      Train: {len(X_tr):,} baris (fraud {int(y_tr.sum())})")
print(f"      Test:  {len(X_te):,} baris (fraud {int(y_te.sum())})")
print(f"\n  Proporsi fraud - Train: {y_tr.mean():.1%} | Test: {y_te.mean():.1%}")

# Hitung scale_pos_weight untuk XGBoost (supaya sensitif terhadap fraud minoritas)
scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f"  Scale pos weight (XGBoost): {scale_pos_weight:.2f}")

[2/6] Membagi train/test (stratified 70/30)...
      Train: 1,481 baris (fraud 81)
      Test:  635 baris (fraud 35)

  Proporsi fraud - Train: 5.5% | Test: 5.5%
  Scale pos weight (XGBoost): 17.28


## Training Random Forest

In [6]:
print("[3/6] Melatih Random Forest (supervised, class_weight=balanced)...")

rf_clf = RandomForestClassifier(**RF_PARAMS)
rf_clf.fit(X_tr, y_tr)

print(f"      Training selesai. Model siap untuk prediksi.")

# Prediksi pada test set untuk evaluasi cepat
print("\n  Performa cepat pada data TEST (Random Forest):")
y_pred_rf = rf_clf.predict(X_te)
print(classification_report(y_te, y_pred_rf, target_names=["normal", "fraud"],
                            digits=3, zero_division=0))

[3/6] Melatih Random Forest (supervised, class_weight=balanced)...
      Training selesai. Model siap untuk prediksi.

  Performa cepat pada data TEST (Random Forest):
              precision    recall  f1-score   support

      normal      0.993     0.998     0.996       600
       fraud      0.969     0.886     0.925        35

    accuracy                          0.992       635
   macro avg      0.981     0.942     0.961       635
weighted avg      0.992     0.992     0.992       635



## Training XGBoost

In [7]:
print("[4/6] Melatih XGBoost (supervised, scale_pos_weight={:.2f})...".format(scale_pos_weight))

if XGBOOST_AVAILABLE:
    # Tambahkan scale_pos_weight ke parameter untuk menangani ketidakseimbangan
    xgb_params = XGB_PARAMS.copy()
    xgb_params['scale_pos_weight'] = scale_pos_weight
    
    xgb_clf = xgb.XGBClassifier(**xgb_params)
    xgb_clf.fit(
        X_tr, y_tr,
        eval_set=[(X_te, y_te)],
        verbose=False
    )
    
    print(f"      Training selesai. Model siap untuk prediksi.")
    
    # Prediksi pada test set untuk evaluasi cepat
    print("\n  Performa cepat pada data TEST (XGBoost):")
    y_pred_xgb = xgb_clf.predict(X_te)
    print(classification_report(y_te, y_pred_xgb, target_names=["normal", "fraud"],
                                digits=3, zero_division=0))
else:
    xgb_clf = None
    print("      [!] XGBoost tidak tersedia - lewati training XGBoost")

[4/6] Melatih XGBoost (supervised, scale_pos_weight=17.28)...
      Training selesai. Model siap untuk prediksi.

  Performa cepat pada data TEST (XGBoost):
              precision    recall  f1-score   support

      normal      0.998     0.998     0.998       600
       fraud      0.971     0.971     0.971        35

    accuracy                          0.997       635
   macro avg      0.985     0.985     0.985       635
weighted avg      0.997     0.997     0.997       635



##  Simpan Kedua Model & Test Split

In [8]:
print("[5/6] Menyimpan model & data test...")

# Buat direktori models jika belum ada
os.makedirs(os.path.dirname(RF_MODEL_PATH), exist_ok=True)

# Simpan Random Forest
joblib.dump(rf_clf, RF_MODEL_PATH)
print(f"      Model RF   -> {RF_MODEL_PATH}")

# Simpan XGBoost (jika tersedia)
if xgb_clf is not None:
    joblib.dump(xgb_clf, XGB_MODEL_PATH)
    print(f"      Model XGB  -> {XGB_MODEL_PATH}")

# Simpan test split agar 03_evaluate.py memakai data yang SAMA persis
joblib.dump({"X_test": X_te, "y_test": y_te}, SPLIT_PATH)
print(f"      Test set   -> {SPLIT_PATH}")

print("\n[OK] Training selesai!")

[5/6] Menyimpan model & data test...
      Model RF   -> c:\Users\gavin\Downloads\NEW Fraudguard\ml_pipeline\models\supervised_rf.pkl
      Model XGB  -> c:\Users\gavin\Downloads\NEW Fraudguard\ml_pipeline\models\supervised_xgb.pkl
      Test set   -> c:\Users\gavin\Downloads\NEW Fraudguard\ml_pipeline\models\test_split.pkl

[OK] Training selesai!


## Feature Importance (Perbandingan RF & XGBoost)

In [9]:
print("[6/6] Analisis Feature Importance...")
print(" PERBANDINGAN FEATURE IMPORTANCE: Random Forest vs XGBoost")

#  Random Forest Feature Importance 
rf_importances = sorted(
    zip(X.columns, rf_clf.feature_importances_),
    key=lambda t: t[1], reverse=True,
)

print("\n  Random Forest Feature Importance:")
print(f"  {'Fitur':30s} {'Pentingnya':>12s} {'Persentase':>12s}")
total_imp_rf = sum(imp for _, imp in rf_importances)
for name, imp in rf_importances:
    pct = (imp / total_imp_rf * 100) if total_imp_rf > 0 else 0
    print(f"  {name:30s} {imp:12.4f} {pct:11.1f}%")

#   XGBoost Feature Importance  
if xgb_clf is not None:
    xgb_importances = sorted(
        zip(X.columns, xgb_clf.feature_importances_),
        key=lambda t: t[1], reverse=True,
    )
    
    print("\n  XGBoost Feature Importance:")
    print(f"  {'Fitur':30s} {'Pentingnya':>12s} {'Persentase':>12s}")
    total_imp_xgb = sum(imp for _, imp in xgb_importances)
    for name, imp in xgb_importances:
        pct = (imp / total_imp_xgb * 100) if total_imp_xgb > 0 else 0
        print(f"  {name:30s} {imp:12.4f} {pct:11.1f}%")
    
    print(" CATATAN: Fitur 'if_anomaly_score' adalah skor anomali Isolation Forest")
    print(" yang dihasilkan dari Lapis 1. Pentingnya fitur menunjukkan seberapa")
    print(" besar kontribusinya dalam keputusan model supervised.\n")
else:
    print("\n[!] XGBoost tidak tersedia - lewati analisis feature importance XGBoost")

[6/6] Analisis Feature Importance...
 PERBANDINGAN FEATURE IMPORTANCE: Random Forest vs XGBoost

  Random Forest Feature Importance:
  Fitur                            Pentingnya   Persentase
  is_refund                            0.3291        32.9%
  if_anomaly_score                     0.2442        24.4%
  refund_count_daily                   0.1272        12.7%
  refund_ratio_daily                   0.1141        11.4%
  amount_zscore_cashier                0.0725         7.3%
  txn_freq_daily                       0.0431         4.3%
  time_gap_seconds                     0.0251         2.5%
  amount_deviation_from_mean           0.0193         1.9%
  amount_rolling_mean_5                0.0179         1.8%
  hour_of_day                          0.0075         0.7%

  XGBoost Feature Importance:
  Fitur                            Pentingnya   Persentase
  is_refund                            0.7680        76.8%
  if_anomaly_score                     0.0930         9.3%
  refund_c